# Разбиение датасета для ручной разметки

В этом ноутбуке собран весь код подготовки: загрузка исходного CSV, назначение ID,
разбиение 80/10/10, распределение train и validation между четырьмя участниками
и проверка совпадения с уже выданными файлами.

## 1. Настройки и зависимости

Перед запуском установите зависимости из корня проекта:
`python3 -m pip install -r requirements.txt`.

Сохраняем параметры исходного разбиения: seed 42, четыре участника и отсутствие
стратификации. Путь к проекту определяется относительно рабочей папки Jupyter.

In [1]:
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
NUMBER_OF_PARTS = 4
MIN_COMMENT_ID = 10_000
MAX_COMMENT_ID = 100_000  # Правая граница не включается.

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data/raw/labeled.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Запустите Jupyter из папки проекта или data/raw")

SOURCE_PATH = ROOT / "data/raw/labeled.csv"
SPLIT_DIR = ROOT / "data/raw/raw_data_splits"

## 2. Загрузка исходного датасета

`comment` — исходный текст; `toxic` — бинарная метка (`0.0` или `1.0`).
Текст не очищается: пробелы, пунктуация и переводы строк сохраняются.
Повторно скачивать датасет не требуется.

In [2]:
source_df = pd.read_csv(SOURCE_PATH)

if list(source_df.columns) != ["comment", "toxic"]:
    raise ValueError("Ожидались колонки comment и toxic в исходном порядке")
if source_df.empty or source_df.isna().any().any():
    raise ValueError("Исходный датасет пуст или содержит пропущенные значения")
if not source_df["toxic"].isin([0.0, 1.0]).all():
    raise ValueError("Колонка toxic должна содержать только 0.0 и 1.0")

print(f"Исходных комментариев: {len(source_df):,}")

Исходных комментариев: 14,412


## 3. Уникальные идентификаторы

Назначаем каждому комментарию пятизначный `comment_id` без повторов.
ID генерируются в исходном порядке строк через `numpy.random.default_rng(42)`.
Изменение seed или порядка исходных строк изменит назначенные ID.

In [3]:
if len(source_df) > MAX_COMMENT_ID - MIN_COMMENT_ID:
    raise ValueError("Пятизначных ID недостаточно для всех комментариев")

rng = np.random.default_rng(seed=RANDOM_STATE)
df = source_df.copy()
df["comment_id"] = rng.choice(
    np.arange(MIN_COMMENT_ID, MAX_COMMENT_ID),
    size=len(df),
    replace=False,
)

## 4. Train, validation и test

Первый вызов оставляет 80% строк в train. Второй делит оставшиеся 20% поровну
между validation и test. Размеры округляются библиотекой: 11 529 / 1 441 / 1 442.
Стратификация не применяется, как и в исходном коде.

In [4]:
train_df, temporary_df = train_test_split(
    df, test_size=0.20, random_state=RANDOM_STATE, shuffle=True,
)
validation_df, test_df = train_test_split(
    temporary_df, test_size=0.50, random_state=RANDOM_STATE, shuffle=True,
)

train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

## 5. Части для четырёх участников

Делим train и validation на четыре последовательные части. Если размер
не делится на четыре, первые части получают по одной дополнительной строке.
Участник N получает `train_part_N.csv` и `validation_part_N.csv`; test остаётся общим.

In [5]:
def split_into_parts(frame, number_of_parts=NUMBER_OF_PARTS):
    """Разделить строки на почти равные части, сохранив их порядок."""
    indices = np.array_split(np.arange(len(frame)), number_of_parts)
    return [frame.iloc[index].reset_index(drop=True) for index in indices]


train_parts = split_into_parts(train_df)
validation_parts = split_into_parts(validation_df)

splits = {}
for group, parts in (("train", train_parts), ("validation", validation_parts)):
    for number, part in enumerate(parts, start=1):
        splits[f"{group}_part_{number}.csv"] = part
splits["test.csv"] = test_df

## 6. Итоговая таблица

Количество строк и распределение исходных меток по каждой части.

In [7]:
summary = pd.DataFrame([
    {
        "Файл": name,
        "Строк": len(frame),
        "Нетоксичных": int(frame["toxic"].eq(0.0).sum()),
        "Токсичных": int(frame["toxic"].eq(1.0).sum()),
    }
    for name, frame in splits.items()
])
summary

,Файл,Строк,Нетоксичных,Токсичных
0,train_part_1.csv,2883,1911,972
1,train_part_2.csv,2882,1917,965
2,train_part_3.csv,2882,1878,1004
3,train_part_4.csv,2882,1936,946
4,validation_part_1.csv,361,237,124
5,validation_part_2.csv,360,246,114
6,validation_part_3.csv,360,239,121
7,validation_part_4.csv,360,239,121
8,test.csv,1442,983,459
